In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from pathlib import Path
import numpy as np
import sys
import importlib
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path().resolve().parent

sys.path.append(str(PROJECT_ROOT / "src"))

import model_utils
importlib.reload(model_utils)
from model_utils import run_nested_cv

In [3]:
sample_fp = PROJECT_ROOT / "outputs" / "sample_points_clean.csv"
samples = pd.read_csv(sample_fp)

samples = samples.dropna(subset="lc")

pred_cols = ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6','B7','B8', 'NDVI', 'NDWI','EVI', 'NDCI', 'AWEIp95', 'NDBI','RR1',
            # 'NDVIre2', 'NDVIre3' ,'AWEIsh'
             ]
metadata_cols = ["label_id","location","class_int", "class_label","obs_date"]


y = samples['lc'].values # values gives you numpy array
X = samples[pred_cols].values
metadata = samples[metadata_cols]
groups = samples['location'].values 


In [6]:
AWEI = "AWEIp95" in pred_cols

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

param_grid = {
    'model__n_estimators': [100,200,500], # have to prefix the keys with the pipeline stepname followed by __
    'model__max_depth': [3,5,10,20],
    'model__min_samples_leaf': [1, 5, 10]
    }

results_df, predictions_df, inner_df = run_nested_cv(X, y, groups, metadata, pipe, param_grid, n_jobs= 4)

results_df["AWEIp95"] = AWEI
inner_df["AWEIp95"] = AWEI
predictions_df["AWEIp95"] = AWEI

model = "RF" + f"_AWEI_{AWEI}"


predictions_fp = PROJECT_ROOT / "outputs" /"RF_outputs"/ f"V2_{model}_predictions_.csv"
predictions_df.to_csv(predictions_fp)
results_fp = PROJECT_ROOT / "outputs"/ "RF_outputs"/ f"V2_{model}_outer_results.csv"
results_df.to_csv(results_fp)
inner_fp = PROJECT_ROOT / "outputs"/"RF_outputs"/ f"V2_{model}_inner_results.csv"
inner_df.to_csv(inner_fp)

F1 Binary:  0.941 +/- 0.028
F1 Macro:   0.941 +/- 0.028
Accuracy:   0.941 +/- 0.028


In [7]:
sample_fp = PROJECT_ROOT / "outputs" / "sample_points_indices.csv"
samples = pd.read_csv(sample_fp)

samples = samples.dropna(subset="lc")

pred_cols = ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6','B7','B8', 'NDVI', 'NDWI','EVI',  'NDCI',  'NDBI','RR1',
             #'AWEIsh','NDVIre2', 'NDVIre3'
             ]
metadata_cols = ["label_id","location","class_int", "class_label","obs_date"]


y = samples['lc'].values # values gives you numpy array
X = samples[pred_cols].values
metadata = samples[metadata_cols]
groups = samples['location'].values 


In [8]:
AWEI = "AWEIp95" in pred_cols

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

param_grid = {
    'model__n_estimators': [100,200,500], # have to prefix the keys with the pipeline stepname followed by __
    'model__max_depth': [3,5,10,20],
    'model__min_samples_leaf': [1, 5, 10]
    }

results_df, predictions_df, inner_df = run_nested_cv(X, y, groups, metadata, pipe, param_grid, n_jobs= 4)

results_df["AWEIp95"] = AWEI
inner_df["AWEIp95"] = AWEI
predictions_df["AWEIp95"] = AWEI

model = "RF" + f"_AWEI_{AWEI}"


predictions_fp = PROJECT_ROOT / "outputs" /"RF_outputs"/ f"V2{model}_predictions_.csv"
predictions_df.to_csv(predictions_fp)
results_fp = PROJECT_ROOT / "outputs"/ "RF_outputs"/ f"V2{model}_outer_results.csv"
results_df.to_csv(results_fp)
inner_fp = PROJECT_ROOT / "outputs"/"RF_outputs"/ f"V2{model}_inner_results.csv"
inner_df.to_csv(inner_fp)

F1 Binary:  0.908 +/- 0.040
F1 Macro:   0.907 +/- 0.040
Accuracy:   0.907 +/- 0.040


In [21]:
results_df

,test_location,test_f1,test_f1_macro,test_accuracy,test_precision,test_recall,test_auroc,best_n_estimators,best_max_depth,best_min_samples_leaf,train_f1,train_f1_macro,train_accuracy,f1_gap,f1_macro_gap,accuracy_gap,AWEIp95
0,Hartbeespoort,0.955815,0.956636,0.956652,0.973113,0.939122,0.956626,100,5,10,0.932676,0.930978,0.931020,-0.023139,-0.025658,-0.025632,False
1,Inle,0.911551,0.907337,0.907529,0.881308,0.943944,0.907176,100,5,5,0.938002,0.936069,0.936127,0.026451,0.028731,0.028598,False
2,Mula,0.841020,0.856842,0.858591,0.922619,0.772682,0.855939,200,10,5,0.966175,0.965554,0.965565,0.125155,0.108711,0.106974,False
3,RawaPening,0.943340,0.943056,0.943057,0.939604,0.947106,0.943053,100,10,1,0.964634,0.963921,0.963935,0.021294,0.020866,0.020878,False
4,Rodman,0.882609,0.852722,0.858787,0.793589,0.994123,0.848914,200,5,10,0.940240,0.939212,0.939229,0.057631,0.086490,0.080442,False
5,Valsequillo,0.944676,0.943996,0.944004,0.941727,0.947644,0.943971,100,5,5,0.934256,0.932616,0.932656,-0.010421,-0.011380,-0.011348,False
6,Vembanad,0.926295,0.926073,0.926074,0.924453,0.928144,0.926072,100,5,10,0.933781,0.931626,0.931694,0.007486,0.005552,0.005620,False
7,Winam,0.855399,0.845258,0.845923,0.806195,0.911000,0.845890,200,5,10,0.941782,0.940402,0.940434,0.086383,0.095144,0.094511,False


In [8]:
# predictions_fp = PROJECT_ROOT / "outputs" / "nested_cv_predictions.csv"

# predictions_df = pd.read_csv(predictions_fp, index_col=0) 
# predictions_df.head()

In [9]:
# conf_matrix = confusion_matrix(predictions_df["y_true"], predictions_df["y_pred"])

# conf_m_disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=["negative","positive"] )
# conf_m_disp.plot()
# plt.show()

In [10]:
# fig, axes = plt.subplots(2, 4, figsize=(16, 8))  # adjust grid to number of locations
# axes = axes.flatten()

# for ax, (location, group) in zip(axes, predictions_df.groupby('location')):
#     cm = confusion_matrix(group['y_true'], group['y_pred'])
#     disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['0', '1'])
#     disp.plot(ax=ax)
#     ax.set_title(location)

# plt.tight_layout()
# plt.show()

In [11]:
# for location, group in predictions_df.groupby('location'):
#     print(f"\n{location}")
#     print(pd.crosstab(group['class_int'], group['y_pred'], normalize='index'))